## BLC data GOLD Framework
NoteBook: gold_calculation_rearc_deltatable

| Author | Date | Change Record | Description |
| ----------- | ----------- | ----------- | -----------
|Hemant kaushik	|05-Nov-2026|	-	| Notebook for ingesting BLC data from SLVR to GOLD
 



In [0]:
from pyspark.sql.functions import (
    avg,col,stddev_pop,count,sum,row_number,min,max,round)
from pyspark.sql.window import Window

In [0]:
##What are the mean and standard deviation of the annual US population across 2013-2018 inclusive


population_df = (
    spark.table("workspace.silver.silver_population")
    .filter(
        (col("year").between(2013, 2018))
        & (col("nation") == "United States")
    )
)

gold_population_stats_df = (
    population_df.agg(
        count("*").alias("year_count"),
        round(avg("population"),2
        ).alias("mean_population"),
        round(stddev_pop("population"),2
        ).alias("population_stddev"),
        min("year").alias("start_year"),
        max("year").alias("end_year")
    )
)

(
    gold_population_stats_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold.gold_population_stats_2013_2018"
    )
)

In [0]:
##For every series_id in the BLS data, what's the best year, meaning the year with the largest sum of value across all its quarters


annual_series_df = (
    spark.table("workspace.silver.silver_pr_data")
    # Keep quarterly records only
    .filter(col("period").isin("Q01","Q02","Q03","Q04"))
    .filter(col("value").isNotNull())
    .groupBy("series_id","year")
    .agg(sum("value").alias("annual_value_sum"))
)

##display(annual_series_df)

best_year_window = (Window.partitionBy("series_id").orderBy(
        col("annual_value_sum").desc(),
        col("year").desc()
    )
)
##display(best_year_window)
best_year_df = (
    annual_series_df.withColumn("_rank",row_number().over(best_year_window)
    )
    .filter(col("_rank") == 1)
    .drop("_rank")
)
##display(best_year_df)

series_label_df = (spark.table("workspace.silver.silver_pr_series")
    .select("series_id"
    ).dropDuplicates(["series_id"])
)

gold_best_year_df = (
    best_year_df
    .join(
        series_label_df,
        on="series_id",
        how="left"
    )
    .select(
        col("series_id"),
        col("year").alias("best_year"),
        col("annual_value_sum")
    )
    .orderBy("series_id")
)

#display(gold_best_year_df)

(
    gold_best_year_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold.gold_bls_best_year_by_series"
    )
)

In [0]:
## For series_id = PRS30006032 and period = Q01, what was the value each year, joined with that year's population where available

bls_q01_df = (
    spark.table("workspace.silver.silver_pr_data")

    .filter(
        (col("series_id") == "PRS30006032")
        & (col("period") == "Q01")
    )

    .select(
        "series_id",
        "year",
        "period",
        "value"
    )
)

population_by_year_df = (
    spark.table("workspace.silver.silver_population")
    .filter(col("nation") == "United States"
    )
    .select("year","population"
    )
    .dropDuplicates(["year"])
)


gold_series_population_df = (
    bls_q01_df.alias("b")
    .join(
        population_by_year_df.alias("p"),
        on="year",
        how="left"
    )
    .select(
        col("b.series_id"),
        col("year"),
        col("b.period"),
        col("b.value"),
        col("p.population")
    )
    .orderBy("year")
)

(
    gold_series_population_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold.gold_prs30006032_q01_population"
    )
)
